# Direct air capture, end to end

trailrunner computes a life cycle inventory by traversing a supply chain of
Python *models* — one per technology — instead of solving a fixed matrix. A
model reads its parameters from a [trailpack](https://github.com/TimoDiepers/trailpack)
parquet file and answers one question: *given this demand, what did I produce,
what do I need, and what did I emit?*

`DirectAirCapture` is the worked example, and it exists because of one number:
the heat needed to regenerate the sorbent is not a constant. Colder, drier air
carries less CO2 and less water to the sorbent per unit of air moved, so heat
and fan work per kilogram captured go up. A coefficient in a table cannot say
that; a function can.

This notebook builds a parameter file, reads it, runs the model on a single
demand, and then lets the orchestrator walk outward from it. Everything runs on
trailrunner's own dependencies — no extras to install beyond `pyarrow`.

## 1. Parameters live in a parquet file, units and all

Model code holds the *behaviour*; the *numbers* come from a parquet file with a
Frictionless `datapackage.json` embedded in its schema metadata. That descriptor
is what makes the columns self-describing: a unit at `fields[].unit.name` and a
[PyST](https://vocab.sentier.dev) concept IRI at `fields[].rdfType`.

In practice trailpack writes this file. Here we write a small one by hand so the
notebook is self-contained: two locations, two years, and the four columns the
DAC model asks for.

In [1]:
import json
import tempfile
from pathlib import Path

import pyarrow as pa
import pyarrow.parquet as pq

ROWS = [
    {"location": "CH", "time": 2020, "heat_demand": 6.0, "electricity_demand": 0.50,
     "temperature": 9.0, "humidity": 0.75},
    {"location": "CH", "time": 2030, "heat_demand": 5.0, "electricity_demand": 0.40,
     "temperature": 10.0, "humidity": 0.70},
    {"location": "RER", "time": 2020, "heat_demand": 6.6, "electricity_demand": 0.55,
     "temperature": 11.0, "humidity": 0.68},
    {"location": "RER", "time": 2030, "heat_demand": 5.5, "electricity_demand": 0.45,
     "temperature": 12.0, "humidity": 0.65},
]

FIELDS = [
    {"name": "location", "type": "string"},
    {"name": "time", "type": "integer", "unit": "year"},
    {"name": "heat_demand", "type": "number", "unit": "MJ",
     "iri": "https://vocab.sentier.dev/parameters/heat-demand"},
    {"name": "electricity_demand", "type": "number", "unit": "kWh",
     "iri": "https://vocab.sentier.dev/parameters/electricity-demand"},
    {"name": "temperature", "type": "number", "unit": "degC",
     "iri": "https://vocab.sentier.dev/parameters/air-temperature"},
    {"name": "humidity", "type": "number", "unit": "dimensionless",
     "iri": "https://vocab.sentier.dev/parameters/relative-humidity"},
]

parameter_file = Path(tempfile.mkdtemp()) / "dac_params.parquet"

table = pa.Table.from_pylist(ROWS)
datapackage = {
    "name": "dac-example-parameters",
    "resources": [
        {
            "name": "parameters",
            "path": parameter_file.name,
            "fields": [
                {
                    "name": field["name"],
                    "type": field["type"],
                    **({"unit": {"name": field["unit"]}} if "unit" in field else {}),
                    **({"rdfType": field["iri"]} if "iri" in field else {}),
                }
                for field in FIELDS
            ],
        }
    ],
}
schema = table.schema.with_metadata({"datapackage.json": json.dumps(datapackage).encode()})
pq.write_table(table.cast(schema), parameter_file)

print(parameter_file)

/var/folders/l1/k90rhb0j0ns58y35ymznsd700000gn/T/tmpz6cwss6s/dac_params.parquet


## 2. Reading parameters, with fallback that says so

`ParameterSet.from_parquet` reads the rows and the descriptor together, so a
column's unit and IRI travel with its value. A lookup goes through
`params.at(location=..., time=...)` and widens until something matches: the exact
row first, then up the `LocationHierarchy`, then linear interpolation between
two bracketing years. Nothing is extrapolated past the data, and every widening
step is written into the row's `provenance`.

In [2]:
from trailrunner import LocationHierarchy, ParameterSet

hierarchy = LocationHierarchy({"CH": "RER", "FR": "RER", "RER": "GLO"})
params = ParameterSet.from_parquet(parameter_file, hierarchy=hierarchy)

row = params.at(location="CH", time=2030)
print(row["heat_demand"], row.unit_of("heat_demand"), row.iri_of("heat_demand"))
print(row.provenance)

5.0 MJ https://vocab.sentier.dev/parameters/heat-demand
{'location_requested': 'CH', 'location_used': 'CH', 'location_fallback': False, 'time_requested': 2030, 'time_used': 2030, 'time_interpolated': False}


France has no row of its own, and 2025 is not a year anybody wrote down. The
lookup still answers — France falls back to `RER`, and the two European rows are
interpolated — but the provenance names both substitutions: `location_used`,
`location_fallback`, `time_interpolated` and the `time_bracket` it interpolated
between. This is the whole contract: widen, but never silently.

In [3]:
fallback = params.at(location="FR", time=2025)
for column in ("heat_demand", "electricity_demand", "temperature", "humidity"):
    print(f"{column:>20}: {fallback[column]:7.3f} {fallback.unit_of(column)}")
print(fallback.provenance)

         heat_demand:   6.050 MJ
  electricity_demand:   0.500 kWh
         temperature:  11.500 degC
            humidity:   0.665 dimensionless
{'location_requested': 'FR', 'location_used': 'RER', 'location_fallback': True, 'time_requested': 2025, 'time_used': 2025, 'time_interpolated': True, 'time_bracket': (2020, 2030)}


## 3. The part that has to be code

The parquet figures assume reference air: 10 °C at 70 % relative humidity.
`ambient_penalty` scales heat and electricity when the air is something else —
colder or drier means more work per kilogram captured, warmer or wetter means
less.

The response is deliberately a plain linear one. The point of the example is
that the dependency lives in code and can be read, argued with and replaced —
not that this particular curve is right.

In [4]:
import inspect

from trailrunner.models import dac

print(inspect.getsource(dac.ambient_penalty))

def ambient_penalty(temperature: float, humidity: float) -> float:
    """Multiplier on heat and electricity demand for non-reference air.

    Colder or drier than the reference gives a value above 1.0; warmer or
    wetter gives one below. Deliberately a simple linear response: the point is
    that the dependency exists and lives in code, not that this particular
    curve is the right one.
    """
    temperature_term = TEMPERATURE_SENSITIVITY * (REFERENCE_TEMPERATURE - temperature)
    humidity_term = HUMIDITY_SENSITIVITY * (REFERENCE_HUMIDITY - humidity)
    return 1.0 + temperature_term + humidity_term



In [5]:
for temperature, humidity in [(10.0, 0.70), (0.0, 0.40), (20.0, 0.90)]:
    penalty = dac.ambient_penalty(temperature, humidity)
    print(f"{temperature:5.1f} degC, RH {humidity:.2f} -> penalty {penalty:.3f}")

 10.0 degC, RH 0.70 -> penalty 1.000
  0.0 degC, RH 0.40 -> penalty 1.190
 20.0 degC, RH 0.90 -> penalty 0.840


## 4. Answering one demand

A model is constructed with its `ParameterSet` and answers a `Demand`: a `Flow`
(what, where, when) plus an amount and a unit. `apply` receives the **full**
demanded amount, never a unit demand — a plant at ten times the scale is not ten
times the plant, and nothing downstream rescales the answer.

The `Result` has three lists:

- **production** — the demand echoed back, same flow, same unit. The Runner
  rejects a model that under-produces rather than letting the inventory shrink.
- **technosphere** — what it needs. These become demands on the queue, here heat
  and electricity at the same place and time, each in its parameter column's own
  unit (MJ and kWh).
- **biosphere** — what it exchanged with the environment. CO2 from air is
  **negative**: this process takes it out of the atmosphere.

In [6]:
from trailrunner import Demand, Flow
from trailrunner.models.dac import CO2_CAPTURED, DirectAirCapture

model = DirectAirCapture(params=params)
demand = Demand(
    flow=Flow(iri=CO2_CAPTURED, location="CH", time=2030), amount=1000.0, unit="kg"
)
result = model.apply(demand)

print("production:")
for exchange in result.production:
    print(f"  {exchange.amount:10.2f} {exchange.unit:4} {exchange.flow.iri}")
print("technosphere:")
for child in result.technosphere:
    print(f"  {child.amount:10.2f} {child.unit:4} {child.flow.iri}")
print("biosphere:")
for exchange in result.biosphere:
    print(f"  {exchange.amount:10.2f} {exchange.unit:4} {exchange.flow.iri}")
print("provenance:", result.provenance)

production:
     1000.00 kg   https://vocab.sentier.dev/products/co2-captured
technosphere:
     5000.00 MJ   https://vocab.sentier.dev/products/heat
      400.00 kWh  https://vocab.sentier.dev/products/electricity
biosphere:
    -1000.00 kg   https://vocab.sentier.dev/flows/co2-from-air
provenance: {'location_requested': 'CH', 'location_used': 'CH', 'location_fallback': False, 'time_requested': 2030, 'time_used': 2030, 'time_interpolated': False}


## 5. The same demand, elsewhere and later

Asking the same 1000 kg in different places and years exercises the parameter
lookup and the penalty together. Switzerland in 2030 sits exactly at reference
conditions, so its penalty is 1.0. Europe is warmer *and* slightly drier: the
warmth dominates, so the penalty lands just below 1.0 even though the underlying
heat figure is higher. France borrows Europe's row — and the `row used` column
says so.

In [7]:
header = f"{'location':>8} {'year':>6} {'degC':>6} {'RH':>5} {'penalty':>8} {'heat [MJ]':>10} {'row used':>10}"
print(header)
for location in ("CH", "FR", "RER"):
    for year in (2020, 2025, 2030):
        flow = Flow(iri=CO2_CAPTURED, location=location, time=year)
        out = model.apply(Demand(flow=flow, amount=1000.0, unit="kg"))
        heat = [d for d in out.technosphere if d.flow.iri == dac.HEAT][0]
        air = params.at(location=location, time=year)
        penalty = dac.ambient_penalty(air["temperature"], air["humidity"])
        print(
            f"{location:>8} {year:>6} {air['temperature']:>6.1f} {air['humidity']:>5.2f} "
            f"{penalty:>8.3f} {heat.amount:>10.1f} {out.provenance['location_used']:>10}"
        )

location   year   degC    RH  penalty  heat [MJ]   row used
      CH   2020    9.0  0.75    0.995     5970.0         CH
      CH   2025    9.5  0.72    0.997     5486.2         CH
      CH   2030   10.0  0.70    1.000     5000.0         CH
      FR   2020   11.0  0.68    0.996     6573.6        RER
      FR   2025   11.5  0.67    0.995     6022.8        RER
      FR   2030   12.0  0.65    0.995     5472.5        RER
     RER   2020   11.0  0.68    0.996     6573.6        RER
     RER   2025   11.5  0.67    0.995     6022.8        RER
     RER   2030   12.0  0.65    0.995     5472.5        RER


## 6. Letting the orchestrator walk outward

A `Glossary` says who produces what; the `Orchestrator` pops a demand, finds its
model, runs it, and pushes the resulting technosphere demands back onto the
queue. With only the DAC model registered, the traversal is one node deep — and
that is exactly what makes the next point visible.

The heat and electricity nobody models are reported as **cutoff leaves**, with a
reason. They are not quietly dropped and not silently zero: the unresolved list
is part of the answer, and it is the to-do list for the next model to write.
Register a boiler that `produces` heat and those 5000 MJ become a node of their
own, with its own emissions.

In [8]:
from trailrunner import Glossary, Orchestrator

report = Orchestrator(Glossary([model])).calculate(demand)

print("inventory:")
for (flow, unit), amount in report.inventory.items():
    print(f"  {amount:10.2f} {unit:4} {flow.iri}  ({flow.location}, {flow.time})")
print("unresolved:")
for record in report.unresolved:
    print(
        f"  {record.demand.amount:10.2f} {record.demand.unit:4} "
        f"{record.demand.flow.iri}  [{record.reason}]"
    )
print("nodes:", len(report.nodes), "truncated:", report.truncated)

inventory:
    -1000.00 kg   https://vocab.sentier.dev/flows/co2-from-air  (CH, 2030)
unresolved:
     5000.00 MJ   https://vocab.sentier.dev/products/heat  [no_model_found]
      400.00 kWh  https://vocab.sentier.dev/products/electricity  [no_model_found]
nodes: 1 truncated: False


Every parameter fallback used along the way is collected per node, so the report
can be audited without re-running anything.

In [9]:
for node_id, provenance in report.provenance.items():
    print(node_id, provenance)

0 {'location_requested': 'CH', 'location_used': 'CH', 'location_fallback': False, 'time_requested': 2030, 'time_used': 2030, 'time_interpolated': False}


## 7. Coverage: outside the data, the model declines

`DirectAirCapture` declares `Coverage(time_range=(2020, 2050))`. Ask it for 2015
and it is not resolved at all — but the report distinguishes *nobody models this*
(`no_model_found`) from *a registered model declined this flow*
(`coverage_excluded`), and names the model in the detail. Those are different
bugs with different fixes: write a model, or widen a coverage.

In [10]:
print(DirectAirCapture.coverage)

early = Demand(
    flow=Flow(iri=CO2_CAPTURED, location="CH", time=2015), amount=1000.0, unit="kg"
)
early_report = Orchestrator(Glossary([model])).calculate(early)

for record in early_report.unresolved:
    print(record.reason)
    print(record.detail)
print("inventory:", early_report.inventory)

Coverage(locations=None, time_range=(2020, 2050))
coverage_excluded
DirectAirCapture declares this product but its coverage does not cover location='CH' time=2015
inventory: {}


## Where this stops

The answer above is an inventory, not a score: trailrunner does no impact
characterization yet, so there is nothing to sum into a single number. Nor is
there a Brightway background — a demand nobody models stays a recorded cutoff.

Two more deliberate boundaries show up in `Report`: every visit is its own node,
never merged with an identical one elsewhere in the tree, which is what keeps
nonlinear models honest; and a cycle is truncated by the depth and node budgets
rather than solved, with `report.truncated` saying when a budget bit.

To go further from here, write a second model — the README's `MyBoiler` is a
ten-line one that produces heat — register it in the `Glossary` alongside
`DirectAirCapture`, and watch the cutoff leaf above turn into a node.